# Neuromorphic Computing Module
## Spiking Neural Networks and Brain-Inspired Computation

This notebook covers:

1. Leaky Integrate-and-Fire (LIF) Neurons
2. Izhikevich Neuron Model
3. Spike-Timing-Dependent Plasticity (STDP)
4. Reservoir Computing with Spiking Networks
5. Pattern Recognition with SNNs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import HTML

print("Neuromorphic Computing Module Loaded")
print("=" * 50)

## 1. Leaky Integrate-and-Fire (LIF) Neuron

The LIF model is the simplest spiking neuron model that captures essential dynamics.

In [ ]:
class LIFNeuron:
    """
    Leaky Integrate-and-Fire Neuron Model
    
    dv/dt = -(v - v_rest)/tau + I/C
    """
    def __init__(self, tau=20, v_rest=-70, v_thresh=-55, v_reset=-75, r=10):
        self.tau = tau        # Membrane time constant (ms)
        self.v_rest = v_rest  # Resting potential (mV)
        self.v_thresh = v_thresh  # Spike threshold (mV)
        self.v_reset = v_reset    # Reset potential (mV)
        self.r = r           # Membrane resistance (MΩ)
        self.v = v_rest      # Current potential
        self.spikes = []     # Spike times
        self.v_history = [] # Voltage history
    
    def step(self, i_input, dt=1):
        """Simulate one time step"""
        # Leaky integration
        dv = (-(self.v - self.v_rest) / self.tau + i_input / 1000) * dt
        self.v += dv
        
        # Check for spike
        if self.v >= self.v_thresh:
            self.spikes.append(1)
            self.v = self.v_reset
        else:
            self.spikes.append(0)
        
        self.v_history.append(self.v)
        return self.v
    
    def reset(self):
        """Reset neuron state"""
        self.v = self.v_rest
        self.spikes = []
        self.v_history = []

# Simulate LIF neuron
neuron = LIFNeuron()
time = np.arange(0, 300, 1)  # 300 ms

# Input: step current injection
i_input = np.zeros(len(time))
i_input[50:150] = 2.5  # Step input 50-150ms
i_input[150:250] = 3.5  # Higher step 150-250ms

# Run simulation
for i in time:
    neuron.step(i_input[i])

# Plot results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(time, i_input, 'g-', linewidth=2)
ax1.set_ylabel('Input Current (mV)')
ax1.set_title('LIF Neuron Response to Step Current')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 5)

ax2.plot(time, neuron.v_history, 'b-', linewidth=1)
ax2.axhline(neuron.v_thresh, color='r', linestyle='--', label='Threshold')
ax2.axhline(neuron.v_reset, color='orange', linestyle=':', label='Reset')
ax2.set_ylabel('Membrane Potential (mV)')
ax2.set_xlabel('Time (ms)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total spikes: {sum(neuron.spikes)}")
print(f"Spike rate: {sum(neuron.spikes) / 0.3:.1f} Hz")

## 2. Izhikevich Neuron Model

The Izhikevich model provides a more biophysically realistic model while remaining computationally efficient.

In [ ]:
class IzhikevichNeuron:
    """
    Izhikevich Neuron Model
    
    dv/dt = 0.04*v² + 5*v + 140 - u + I
    du/dt = a*(b*v - u)
    
    When v >= 30: v = c, u = u + d
    """
    def __init__(self, a=0.02, b=0.2, c=-65, d=2):
        self.a = a  # Recovery time constant
        self.b = b  # Recovery sensitivity
        self.c = c  # Reset voltage
        self.d = d  # Recovery reset
        self.v = c  # Initial voltage
        self.u = b * c  # Initial recovery
    
    def step(self, i_input, dt=1):
        """Simulate one time step using RK4"""
        for _ in range(int(dt)):
            v, u = self.v, self.u
            
            # Membrane potential update
            dv = 0.04 * v**2 + 5*v + 140 - u + i_input
            du = self.a * (self.b * v - u)
            
            v += dv
            u += du
            
            # Spike reset
            if v >= 30:
                v = self.c
                u = u + self.d
            
            self.v = v
            self.u = u
        
        return self.v

# Different neuron behaviors
regimes = {
    'RS (Regular Spiking)': {'a': 0.02, 'b': 0.2, 'c': -65, 'd': 2},
    'FS (Fast Spiking)': {'a': 0.1, 'b': 0.2, 'c': -65, 'd': 2},
    'IB (Intrinsically Bursting)': {'a': 0.02, 'b': 0.2, 'c': -55, 'd': 4},
    'CH (Chattering)': {'a': 0.02, 'b': 0.2, 'c': -50, 'd': 2},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

time = np.arange(0, 200, 0.5)

for idx, (name, params) in enumerate(regimes.items()):
    neuron = IzhikevichNeuron(**params)
    voltages = []
    
    # Input: step current
    i_input = 10 if 'CH' not in name else 7
    
    for t in time:
        v = neuron.step(i_input)
        voltages.append(v)
    
    ax = axes[idx]
    ax.plot(time, voltages, 'b-', linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Membrane Potential (mV)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Izhikevich Neuron Regimes', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Spike-Timing-Dependent Plasticity (STDP)

STDP is the biological learning rule where synaptic strength changes based on spike timing.

In [ ]:
def stdp(delta_t, a_plus=0.01, a_minus=0.012, tau_plus=20, tau_minus=20):
    """
    Calculate STDP weight change.
    
    Parameters:
    -----------
    delta_t : float
        Time difference: t_post - t_pre (ms)
    """
    if delta_t >= 0:
        # Pre before post (or simultaneous): potentiation
        return a_plus * np.exp(-delta_t / tau_plus)
    else:
        # Post before pre: depression
        return -a_minus * np.exp(delta_t / tau_minus)

# Visualize STDP curve
delta_t_range = np.linspace(-100, 100, 1000)

fig, ax = plt.subplots(figsize=(12, 6))

weight_changes = [stdp(dt) for dt in delta_t_range]

ax.plot(delta_t_range, weight_changes, 'b-', linewidth=2)
ax.axhline(0, color='gray', linestyle='-', alpha=0.5)
ax.axvline(0, color='gray', linestyle='-', alpha=0.5)

# Fill regions
ax.fill_between(delta_t_range[delta_t_range >= 0], 0, 
                [weight_changes[i] for i in range(len(delta_t_range)) if delta_t_range[i] >= 0],
                alpha=0.3, color='green', label='Potentiation (Δt > 0)')
ax.fill_between(delta_t_range[delta_t_range < 0], 0, 
                [weight_changes[i] for i in range(len(delta_t_range)) if delta_t_range[i] < 0],
                alpha=0.3, color='red', label='Depression (Δt < 0)')

# Annotations
ax.annotate('Pre before Post', xy=(40, 0.008), fontsize=12, ha='center')
ax.annotate('Post before Pre', xy=(-40, -0.008), fontsize=12, ha='center')

ax.set_xlabel('Δt = t_post - t_pre (ms)', fontsize=12)
ax.set_ylabel('Weight Change (Δw)', fontsize=12)
ax.set_title('STDP Learning Window', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.015, 0.015)

plt.tight_layout()
plt.show()

In [ ]:
# Simulate STDP learning
class STDPLearning:
    def __init__(self, w_init=0.5, w_max=1.0, w_min=0.0):
        self.w = w_init
        self.w_max = w_max
        self.w_min = w_min
    
    def update(self, delta_t):
        dw = stdp(delta_t)
        self.w = np.clip(self.w + dw, self.w_min, self.w_max)
        return self.w

# Simulate weight evolution
np.random.seed(42)
stdp_sim = STDPLearning()
weights = [stdp_sim.w]

# Generate random spike timing differences
for _ in range(500):
    delta_t = np.random.uniform(-80, 80)
    w = stdp_sim.update(delta_t)
    weights.append(w)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(weights, 'b-', alpha=0.7)
ax.axhline(stdp_sim.w_max, color='g', linestyle='--', label='Max Weight')
ax.axhline(stdp_sim.w_min, color='r', linestyle='--', label='Min Weight')
ax.set_xlabel('Learning Steps')
ax.set_ylabel('Synaptic Weight')
ax.set_title('STDP Weight Evolution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial weight: {weights[0]:.3f}")
print(f"Final weight: {weights[-1]:.3f}")

## 4. Reservoir Computing with Spiking Networks

Reservoir computing leverages the dynamics of a spiking network for temporal computation.

In [ ]:
class SpikingReservoir:
    """
    Simple spiking reservoir for temporal pattern recognition
    """
    def __init__(self, n_neurons=100, sparsity=0.1, spectral_radius=0.9):
        self.n = n_neurons
        self.sparsity = sparsity
        
        # Create sparse weight matrix
        W = np.random.randn(n_neurons, n_neurons)
        mask = np.random.rand(n_neurons, n_neurons) < sparsity
        W = W * mask
        
        # Scale to desired spectral radius
        eigenvalues = np.linalg.eigvals(W)
        W = W * (spectral_radius / np.max(np.abs(eigenvalues)))
        
        self.W = W
        self.state = np.zeros(n_neurons)
        self.neurons = [LIFNeuron() for _ in range(n_neurons)]
    
    def step(self, input_current, dt=1):
        """Process one time step"""
        # Compute recurrent input
        recurrent = self.W @ self.state
        
        # Total input
        total_input = input_current + recurrent * 0.1
        
        # Update all neurons
        for i, neuron in enumerate(self.neurons):
            neuron.step(total_input[i] / self.n)
            self.state[i] = neuron.v / 100  # Normalized state
        
        return self.state.copy()
    
    def reset(self):
        self.state = np.zeros(self.n)
        for neuron in self.neurons:
            neuron.reset()

# Demonstrate reservoir dynamics
reservoir = SpikingReservoir(n_neurons=50)

time = np.arange(0, 500, 1)
states_history = []

# Input: oscillating signal
input_signal = 2 * np.sin(2 * np.pi * time / 50) + 3

for i in time:
    state = reservoir.step(input_signal[i])
    states_history.append(state)

states_history = np.array(states_history)

# Plot reservoir dynamics
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Input signal
axes[0].plot(time, input_signal, 'g-', linewidth=1)
axes[0].set_ylabel('Input')
axes[0].set_title('Reservoir Input')
axes[0].grid(True, alpha=0.3)

# Sample neuron activities
for i in range(5):
    axes[1].plot(time, states_history[:, i*10], alpha=0.7)
axes[1].set_ylabel('Neuron Activity')
axes[1].set_title('Sample Neuron States')
axes[1].grid(True, alpha=0.3)

# PCA of reservoir states
from scipy.linalg import eigh
cov = np.cov(states_history.T)
eigenvalues, eigenvectors = eigh(cov)

pc1 = states_history @ eigenvectors[:, -1]
pc2 = states_history @ eigenvectors[:, -2]

scatter = axes[2].scatter(pc1, pc2, c=time, cmap='viridis', s=1, alpha=0.5)
axes[2].set_xlabel('PC1')
axes[2].set_ylabel('PC2')
axes[2].set_title('Reservoir State Space (PCA)')
plt.colorbar(scatter, ax=axes[2], label='Time (ms)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Pattern Recognition with SNNs

Spiking neural networks can learn to recognize patterns through spike timing.

In [ ]:
def poisson_spike_generator(rate, duration, dt=1):
    """
    Generate Poisson distributed spikes.
    """
    n_steps = int(duration / dt)
    spikes = np.random.rand(n_steps) < (rate * dt / 1000)
    return spikes

def classify_pattern(spike_trains, templates):
    """
    Simple template matching classifier using spike coincidence.
    """
    best_match = None
    best_score = -np.inf
    
    for label, template in templates.items():
        # Count synchronous spikes
        coincidence = np.sum(spike_trains * template)
        if coincidence > best_score:
            best_score = coincidence
            best_match = label
    
    return best_match, best_score

# Create templates for two patterns
np.random.seed(42)
n_neurons = 100
n_time_steps = 100

templates = {
    'Pattern A': np.random.rand(n_neurons, n_time_steps) > 0.9,  # Sparse
    'Pattern B': np.random.rand(n_neurons, n_time_steps) > 0.7,  # Dense
}

# Generate test patterns
test_A = np.random.rand(n_neurons, n_time_steps) > 0.9
test_B = np.random.rand(n_neurons, n_time_steps) > 0.7

# Classify
result_A, score_A = classify_pattern(test_A, templates)
result_B, score_B = classify_pattern(test_B, templates)

print(f"Test Pattern 1 classified as: {result_A}")
print(f"Test Pattern 2 classified as: {result_B}")
print(f"Classification scores: {score_A:.0f}, {score_B:.0f}")

In [ ]:
# Visualize pattern classification
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Templates
axes[0, 0].imshow(templates['Pattern A'][:30], aspect='auto', cmap='Greys')
axes[0, 0].set_title('Template A')
axes[0, 0].set_ylabel('Neuron')

axes[0, 1].imshow(templates['Pattern B'][:30], aspect='auto', cmap='Greys')
axes[0, 1].set_title('Template B')

# Test patterns
axes[1, 0].imshow(test_A[:30], aspect='auto', cmap='Greys')
axes[1, 0].set_title(f'Input 1 → {result_A}')
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_ylabel('Neuron')

axes[1, 1].imshow(test_B[:30], aspect='auto', cmap='Greys')
axes[1, 1].set_title(f'Input 2 → {result_B}')
axes[1, 1].set_xlabel('Time')

# Raster plot summary
ax3 = axes[0, 2]
for i in range(20):
    spike_times = np.where(test_A[i])[0]
    ax3.scatter(spike_times, [i]*len(spike_times), s=1, c='blue')
ax3.set_title('Input 1 Raster')
ax3.set_xlabel('Time')
ax3.set_ylabel('Neuron')

ax4 = axes[1, 2]
for i in range(20):
    spike_times = np.where(test_B[i])[0]
    ax4.scatter(spike_times, [i]*len(spike_times), s=1, c='red')
ax4.set_title('Input 2 Raster')
ax4.set_xlabel('Time')
ax4.set_ylabel('Neuron')

plt.tight_layout()
plt.show()

print("\n" + "=" * 50)
print("Neuromorphic Computing Module Complete")
print("=" * 50)
print("\nKey Concepts Covered:")
print("  • LIF and Izhikevich neuron models")
print("  • STDP learning rule")
print("  • Reservoir computing dynamics")
print("  • SNN-based pattern recognition")